In [6]:
import os, sys
week1_path = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
sys.path.insert(0,week1_path)
# import scraper.py
from scraper import fetch_website_contents, fetch_website_links
# import environment variables module
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from openai import OpenAI
openai = OpenAI()

In [7]:
# Load environment variables
load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
elif api_key.strip() != api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")
    

API key found and looks good so far!


In [8]:
links = fetch_website_links("https://edwarddonner.com")
links

['https://edwarddonner.com/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2025/09/15/ai-in-production-gen-ai-and-agentic-ai-on-

# Get relevant links to the website and absolute links (replace relative links) using AI

In [11]:
link_system_prompt = """You are a helpful assistant for extracting links from a website. 
You will be given a list of links, and you should extract the ones that are relevant to sales brochures about the company,
such as links to an About page, a Products or Services page, or any page that seems to be focused on marketing the company's offerings. 
Please respond in a JSON format like below example, with a list of relevant links and their descriptions if available. 
{
    "links": [
        {
            "type": "About us page with company history and mission statement",
            "url": "https://example.com/about"
            
        },
        {
            "type": "Products page showcasing our offerings and features",
            "url": "https://example.com/products"
        }
    ]
}
"""


In [15]:
#function to extract relevant links from a website for user prompt.
def get_links_user_prompt(url):
    user_prompt = f"""Here is a list of links from the website {url}:
    Please respond in JSON format with a list of relevant links and their descriptions if available.
    Respond with a full https url for each link.
    Do not include privacy, terms of service, or any other irrelevant links."""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [17]:
print(get_links_user_prompt("https://edwarddonner.com"))

Here is a list of links from the website https://edwarddonner.com:
    Please respond in JSON format with a list of relevant links and their descriptions if available.
    Respond with a full https url for each link.
    Do not include privacy, terms of service, or any other irrelevant links.https://edwarddonner.com/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.com/20

In [25]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model="gpt-5-nano",
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
        #This forces the model to output only valid JSON, preventing malformed or partially‑structured responses. 
        #Even if the system prompt asks for JSON, the model may not always comply, so JSON mode adds a guaranteed enforcement layer. This is an inference‑time parsing feature where the API validates the output and ensures it is valid JSON before returning it.
        #JSON mode guarantees valid JSON, but does not enforce a schema — for schema‑validated output, use Structured Outputs.  
    )
    return response.choices[0].message.content

In [26]:
print(select_relevant_links("https://edwarddonner.com"))

{
  "links": [
    {
      "type": "Company homepage overview and offerings",
      "url": "https://edwarddonner.com/",
      "description": "Homepage providing overview of Edward Donner's work and offerings."
    },
    {
      "type": "About page with founder background and Nebula association",
      "url": "https://edwarddonner.com/about-me-and-about-nebula/",
      "description": "About the founder and Nebula affiliation, including company background."
    },
    {
      "type": "Curriculum page for training programs",
      "url": "https://edwarddonner.com/curriculum/",
      "description": "Curriculum page outlining educational programs and courses."
    },
    {
      "type": "Proficient page describing capabilities and services",
      "url": "https://edwarddonner.com/proficient/",
      "description": "Page detailing capabilities and services offered."
    }
  ]
}
